# Preprocessing pipeline

IMP-06. Runs the code in src/preprocessing.py.

Stage A applies the rule-based decisions from docs/cleaning_decisions.xlsx once, before any split. Stage B (imputation, missing indicators, log transform, scaling) is returned unfitted, so it can be fitted inside cross-validation on the training folds only.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))          # lets the notebook import src/preprocessing.py

from src import preprocessing as pp

SEED = 42
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 90)

## Inputs

Only decisions with status approved or proposed are applied.

In [2]:
df = pp.load_raw(ROOT)
groups = pp.load_feature_groups(ROOT)
all_decisions = pp.load_decisions(ROOT, statuses=("approved", "proposed", "rejected"))
decisions = pp.load_decisions(ROOT)          # the ones that will be applied

print(f"Raw data: {df.shape[0]} records x {df.shape[1]} columns")
print(f"Feature groups: {len(groups)} variables, {int((groups['group'] == 'excluded').sum())} excluded")
pd.crosstab(all_decisions["action"], all_decisions["status"], margins=True, margins_name="total")

Raw data: 541 records x 45 columns
Feature groups: 41 variables, 3 excluded


status,proposed,total
action,,
correct,1,1
keep,3,3
recode,1,1
recompute,2,2
set_missing,26,26
total,33,33


## Stage A: cleaning

In [3]:
X, y, ids, change_log = pp.clean(df, groups, decisions)

print(f"Model features: {X.shape[1]} (41 features minus {int((groups['group'] == 'excluded').sum())} excluded)")
print(f"Records: {X.shape[0]} | PCOS: {int(y.sum())} | not PCOS: {int((y == 0).sum())}")
print(f"Values changed: {len(change_log)}")
change_log

Model features: 38 (41 features minus 3 excluded)
Records: 541 | PCOS: 177 | not PCOS: 364
Values changed: 31


,decision,patient_file_no,column,before,after,rule
0,D01,224,Pulse rate(bpm),18,NaN,set to missing
1,D02,297,Pulse rate(bpm),13.0,NaN,set to missing
2,D03,162,BP _Systolic (mmHg),12,NaN,set to missing
3,D04,201,BP _Diastolic (mmHg),8,NaN,set to missing
4,D05,330,FSH(mIU/mL),5052.0,NaN,set to missing
5,D06,456,LH(mIU/mL),2018.0,NaN,set to missing
6,D07,192,Vit D3 (ng/mL),6014.66,NaN,set to missing
7,D08,196,Vit D3 (ng/mL),5418.6,NaN,set to missing
8,D10,513,Cycle(R/I),5,NaN,set to missing
9,D12,124,II beta-HCG(mIU/mL),1.99.,1.99,corrected


## Checks after cleaning

In [4]:
excluded = groups.loc[groups["group"] == "excluded", "variable"].tolist()
set_missing = decisions[(decisions["scope"] == "cell") & (decisions["action"] == "set_missing")]
row = dict(zip(ids, X.index))

checks = {
    "All features are numeric": all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns),
    "No excluded, ID or label columns in X": not (set(X.columns) & set(excluded + ["Sl. No", pp.ID, pp.LABEL])),
    "Cycle(R/I) contains only 0, 1 or missing": set(X["Cycle(R/I)"].dropna().unique()) <= {0, 1},
    "Every 'set_missing' decision is now missing": all(
        pd.isna(X.at[row[int(d.patient_file_no)], d.column]) for d in set_missing.itertuples() if d.column in X.columns),
    "Label has 177 PCOS and 364 not PCOS": (int(y.sum()), int((y == 0).sum())) == (177, 364),
    "Cleaning is deterministic (same result twice)": X.equals(pp.clean(df, groups, decisions)[0]),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

PASS  All features are numeric
PASS  No excluded, ID or label columns in X
PASS  Cycle(R/I) contains only 0, 1 or missing
PASS  Every 'set_missing' decision is now missing
PASS  Label has 177 PCOS and 364 not PCOS
PASS  Cleaning is deterministic (same result twice)


## Missing values after cleaning

In [5]:
missing = X.isna().sum()
missing[missing > 0].to_frame("missing values")

,missing values
Pulse rate(bpm),2
Cycle(R/I),1
Marraige Status (Yrs),1
FSH(mIU/mL),1
LH(mIU/mL),1
FSH/LH,2
AMH(ng/mL),1
Vit D3 (ng/mL),3
Fast food (Y/N),1
BP _Systolic (mmHg),1


## Stage B by model family

linear = LR/SVM (impute, log-transform blood-test values, scale), tree = RF (impute), native = XGBoost/CatBoost (nothing; they handle NaN). Fitted here on a demo split only.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

rows = []
for family in ["linear", "tree", "native"]:
    pipe = pp.build_preprocessor(groups, family).fit(X_train)
    out = pipe.transform(X_test)
    rows.append({"family": family, "columns_out": out.shape[1],
                 "missing_indicators": sum(c.startswith("missingindicator_") for c in out.columns),
                 "missing_values_left": int(out.isna().sum().sum())})
pd.DataFrame(rows)

,family,columns_out,missing_indicators,missing_values_left
0,linear,51,13,0
1,tree,51,13,0
2,native,38,0,5


## Stage B uses training data only

The AMH fill value should match the training median, not the median of all 541 records.

In [7]:
pipe = pp.build_preprocessor(groups, "tree").fit(X_train)
imputer = pipe.named_steps["preprocess"].named_transformers_["numeric"]
position = list(imputer.feature_names_in_).index("AMH(ng/mL)")

pd.DataFrame({"AMH median": {
    "learned by the pipeline": imputer.statistics_[position],
    "training split only": X_train["AMH(ng/mL)"].median(),
    "all 541 records (must NOT be used)": X["AMH(ng/mL)"].median(),
}})

,AMH median
learned by the pipeline,3.75
training split only,3.75
all 541 records (must NOT be used),3.70


## Preprocessing steps

Stage A (done once, before splitting the data):

1. Removed extra spaces from the column names.
2. Applied the cell decisions from cleaning_decisions.xlsx: wrong values set to missing, the typo '1.99.' corrected, and AMH 66 kept.
3. Turned every feature into a number. Anything that wasn't a number became missing.
4. Recalculated BMI for patient 441 from weight and height.
5. Recalculated FSH/LH from the cleaned FSH and LH.
6. Changed cycle regularity from 2/4 to 0/1.
7. Removed the excluded features (blood group, beta-HCG I and II), the IDs, the label and the empty extra column. This leaves 38 features.

In total, 31 values were changed and all 6 checks passed.

Stage B (fitted only on the training data, inside cross-validation):

8. Filled missing values with the median (the most common value for yes/no columns), and added a column showing where a value was missing.
9. For Logistic Regression and SVM, took the log of the blood test values, because they're skewed, and scaled the numeric variables.
10. Random Forest only uses the filling step. XGBoost and CatBoost get no Stage B steps, because they handle missing values themselves.

I checked that the AMH fill value (3.75) is the median of the training data and not of all the data (3.70). This shows the test data isn't leaking into training.